# Competencia: ¿qué se ve desde el satélite?

**Sesión 3 · IA Intermedio · EstudIA 2026**

Cada imagen es un cuadro de unos 640 × 640 metros de Europa, fotografiado por el satélite Sentinel-2
y reducido a 32 × 32 píxeles. Tu red tiene que decir **qué uso tiene el suelo**:

| Etiqueta | Qué hay |
|---|---|
| `cultivo_anual` | Cultivos que se siembran y cosechan cada año |
| `cultivo_permanente` | Cultivos que duran años: frutales, viñedos, olivares |
| `pastizal` | Pasto para el ganado |
| `vegetacion_herbacea` | Hierbas y arbustos silvestres |
| `bosque` | Bosque |
| `rio` | Río |
| `mar_o_lago` | Mar o lago |
| `autopista` | Autopista o vía principal |
| `residencial` | Zona de vivienda |
| `industrial` | Zona industrial o comercial |

**Reglas**

1. Usa una **CNN en PyTorch entrenada desde cero**: nada de modelos preentrenados. Todo lo demás lo
   puedes cambiar.
2. Tienes 2.500 imágenes con etiqueta (250 por clase) y 3.000 sin etiqueta. Separa tu propia
   validación: no tienes las etiquetas de prueba.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor
   **exactitud**. Las diez clases tienen el mismo número de imágenes.
4. La tabla en vivo mide tu exactitud con **la mitad** de las imágenes de prueba. Al final, el
   profesor revela el resultado con **la otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Si Colab te da GPU, actívala (*Entorno de
ejecución › Cambiar tipo de entorno de ejecución › GPU T4*); sin GPU también funciona, solo más
despacio. Ejecuta todo el notebook tal como está (*Entorno de ejecución › Ejecutar todas*). Cuando te
pida la clave, cópiala de la página de la competencia en la plataforma y pégala. Así haces tu primer
envío. Después, mejora tu modelo.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, time, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s03")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

GPU = torch.cuda.is_available()
DISPOSITIVO = torch.device("cuda" if GPU else "cpu")

def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

datos = np.load(descargar("satelite_entrenamiento.npz"))
prueba = np.load(descargar("satelite_prueba.npz"))
imagenes, etiquetas = datos["imagenes"], datos["etiquetas"]
ids_prueba, imagenes_prueba = prueba["ids"], prueba["imagenes"]
CLASES = sorted(set(etiquetas.tolist()))
print(f"Listo: {miles(len(imagenes))} imágenes de entrenamiento y {miles(len(imagenes_prueba))} de prueba, "
      f"de {imagenes.shape[1]} × {imagenes.shape[2]} píxeles.",
      f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: funciona igual, solo más despacio.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error, urllib.request

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: los ids de las imágenes de prueba. predicciones: una etiqueta por imagen, como "bosque".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s03/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    print(f"Envío {res['envio']} recibido. Exactitud en la tabla: {res['exactitud']:.1%}.")
    print(f"Tu mejor resultado: {res['mejor']:.1%} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")

In [ ]:
# @title Función para dibujar las curvas de entrenamiento
def curvas(historia, titulo):
    """Pérdida y exactitud por época, en entrenamiento y en validación."""
    epocas = range(1, len(historia["loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
    for eje, metrica, nombre in ((ejes[0], "loss", "Pérdida"), (ejes[1], "accuracy", "Exactitud")):
        eje.plot(epocas, historia[metrica], label="Entrenamiento", color="#000000", lw=2)
        eje.plot(epocas, historia["val_" + metrica], label="Validación", color="#CB318E", lw=2)
        eje.set_xlabel("Época"); eje.set_title(nombre); eje.grid(alpha=0.3)
        eje.spines[["top", "right"]].set_visible(False)
    mejor = int(np.argmin(historia["val_loss"]))
    ejes[0].axvline(mejor + 1, color="#7FC200", ls="--", lw=1.5, label=f"Menor pérdida de validación (época {mejor + 1})")
    ejes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()

---
## 1. Los datos

In [ ]:
fig, ejes = plt.subplots(5, 10, figsize=(14, 7.6))
for c, clase in enumerate(CLASES):
    for f, i in enumerate(np.where(etiquetas == clase)[0][:5]):
        ejes[f, c].imshow(imagenes[i]); ejes[f, c].axis("off")
    ejes[0, c].set_title(clase.replace("_", " "), fontsize=9)
plt.tight_layout(); plt.show()

### 1.1 Entrenamiento y validación

Separamos el 20 % como validación, con el mismo número de imágenes por clase. Pasamos todo a tensores
de PyTorch y, si hay GPU, a la GPU. Los colores se normalizan con la media y la desviación del
entrenamiento.

In [ ]:
X = torch.from_numpy(imagenes).permute(0, 3, 1, 2).contiguous()       # (n, 3 canales, 32, 32)
y = torch.tensor([CLASES.index(e) for e in etiquetas])
i_ent, i_val = train_test_split(np.arange(len(X)), test_size=0.2, stratify=y.numpy(), random_state=42)
X_ent, y_ent = X[i_ent].to(DISPOSITIVO), y[i_ent].to(DISPOSITIVO)
X_val, y_val = X[i_val].to(DISPOSITIVO), y[i_val].to(DISPOSITIVO)
X_prueba = torch.from_numpy(imagenes_prueba).permute(0, 3, 1, 2).contiguous().to(DISPOSITIVO)

MEDIA = (X_ent.float() / 255).mean(dim=(0, 2, 3), keepdim=True)
DESV = (X_ent.float() / 255).std(dim=(0, 2, 3), keepdim=True)

def normalizar(x):
    return (x.float() / 255 - MEDIA) / DESV

print(f"Entrenamiento: {miles(len(X_ent))} · Validación: {miles(len(X_val))} · Prueba: {miles(len(X_prueba))}")

---
## 2. Tu modelo

Las piezas que puedes cambiar:

- `crear_modelo()`: la arquitectura de la red.
- `EPOCAS`, `LOTE` y `TASA`: cuántas épocas entrenar, cuántas imágenes por lote y la tasa de
  aprendizaje del optimizador.
- `entrenar(...)`: el ciclo de entrenamiento.

Así como vienen funcionan, pero el modelo tiene problemas. Ejecútalas tal cual para tu primer envío y
después mejóralas.

In [ ]:
def crear_modelo():
    return nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 32 × 32 -> 16 × 16
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 16 × 16 -> 8 × 8
        nn.Flatten(),                                                               # 64 × 8 × 8 = 4.096
        nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
        nn.Linear(128, len(CLASES)),
    )

EPOCAS = 30
LOTE = 64
TASA = 1e-3

In [ ]:
@torch.no_grad()
def evaluar(modelo, X, y):
    modelo.eval()
    salida = torch.cat([modelo(normalizar(X[i:i + 500])) for i in range(0, len(X), 500)])
    return F.cross_entropy(salida, y).item(), (salida.argmax(1) == y).float().mean().item()

def entrenar(modelo, X, y, epocas, X_val=None, y_val=None):
    """Entrena `modelo` con X, y. Si le das validación, la mide al final de cada época."""
    torch.manual_seed(7)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=TASA)
    historia = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    inicio = time.time()
    for epoca in range(epocas):
        modelo.train()
        orden = torch.randperm(len(X), device=X.device)
        for i in range(0, len(X), LOTE):
            idx = orden[i:i + LOTE]
            perdida = F.cross_entropy(modelo(normalizar(X[idx])), y[idx])
            optimizador.zero_grad()
            perdida.backward()
            optimizador.step()
        if X_val is not None:
            for k, v in zip(historia, evaluar(modelo, X, y) + evaluar(modelo, X_val, y_val)):
                historia[k].append(v)
    print(f"{epocas} épocas en {time.time() - inicio:.0f} s.")
    return historia

@torch.no_grad()
def predecir(modelo, X):
    modelo.eval()
    return torch.cat([modelo(normalizar(X[i:i + 500])).argmax(1) for i in range(0, len(X), 500)]).cpu()

### 2.1 Entrenar y validar

In [ ]:
torch.manual_seed(7)
modelo = crear_modelo().to(DISPOSITIVO)
historia = entrenar(modelo, X_ent, y_ent, EPOCAS, X_val, y_val)
curvas(historia, f"{miles(sum(p.numel() for p in modelo.parameters()))} parámetros")
print(f"Última época · exactitud de entrenamiento: {historia['accuracy'][-1]:.1%} · validación: {historia['val_accuracy'][-1]:.1%}")

### 2.2 Predecir la prueba y enviar

Cambia la nota para recordar qué probaste en cada envío. La primera vez te pide la clave.

In [ ]:
predicciones = [CLASES[i] for i in predecir(modelo, X_prueba)]
enviar(ids_prueba, predicciones, nota="primer envío")

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud de **validación** y envía solo cuando mejore.

<details><summary>Pista 1</summary>

Mira las curvas del primer modelo: ¿qué le pasa a la exactitud de entrenamiento? ¿Y a la de validación?
¿Qué vimos en la demostración cuando pasó lo mismo?
</details>

<details><summary>Pista 2</summary>

Aumento de datos. ¿Qué le puedes hacer a una foto tomada desde un satélite sin cambiar lo que hay en
el suelo? En la demostración volteamos las fotos como en un espejo. Aquí también puedes voltearlas de
arriba abajo o girarlas 90°: un bosque visto desde el cielo no tiene un “arriba”. Escribe una función
que altere cada lote dentro de `entrenar`, justo antes de pasarlo al modelo, y úsala solo en el
entrenamiento. `torch.flip`, `torch.rot90` y `torch.where` te sirven.
</details>

<details><summary>Pista 3</summary>

La arquitectura. Compara `crear_modelo` con la CNN de la demostración: ¿cuántas convoluciones hay por
bloque? ¿Usa `nn.BatchNorm2d`? Mira también dónde están casi todos los parámetros: la capa
`nn.Linear(4096, 128)` tiene más de medio millón. `nn.AdaptiveAvgPool2d(1)` resume cada mapa en un solo
número y deja una cabeza mucho más pequeña. `nn.Dropout` también ayuda.
</details>

<details><summary>Pista 4</summary>

Con aumento de datos, la red necesita más épocas para aprender. Cuando ya elegiste la arquitectura y el
número de épocas, puedes entrenar el modelo final con las 2.500 imágenes (entrenamiento más validación)
antes de predecir la prueba. Si tienes GPU, aprovéchala.
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo
a esa mitad de la prueba. El resultado final se calcula con la otra mitad.

**Fuente de los datos:** Helber, P., Bischke, B., Dengel, A. y Borth, D. (2019). EuroSAT: A Novel
Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification. *IEEE Journal of
Selected Topics in Applied Earth Observations and Remote Sensing*. Licencia MIT. Imágenes de
Copernicus Sentinel-2 (ESA). Tomamos una muestra con el mismo número de imágenes por clase, las
redujimos de 64 × 64 a 32 × 32 píxeles y renombramos las clases al español.